# Tutorial 02: Data Preprocessing and Real Data Integration

Learn how to work with real market data, handle missing values, and preprocess data for factor calculation.

## What You'll Learn
- Connect to real data sources
- Handle missing data and outliers
- Universe filtering and stock selection
- Data quality checks
- PIT (Point-in-Time) data handling

In [ ]:
import sys
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

sys.path.insert(0, '/home/shw/quant_projects/factor_engine')
sys.path.insert(0, '/home/shw/quant_projects/notebooks')

from utils import (
    display_success, display_warning, display_dataframe_summary,
    display_metrics, NotebookTimer, ProgressBar
)

## Step 1: Understanding Data Sources

The factor_engine supports multiple data backends:
- **Pandas**: In-memory DataFrames
- **Polars**: High-performance columnar processing
- **DuckDB**: SQL-based queries on large datasets

In [ ]:
from api import col, rank, ts_mean, Factor
from backend.pandas_backend import PandasBackend
from storage.datasource import DataSource
from runtime.engine import FactorEngine

display_success("Imported data handling components")

## Step 2: Create Sample Market Data

Let's simulate realistic market data with common issues: missing values, outliers, gaps.

In [ ]:
def generate_sample_market_data(n_stocks=50, n_days=252):
    """Generate realistic sample market data with common data issues."""
    np.random.seed(42)
    
    dates = pd.date_range(end=datetime.now(), periods=n_days, freq='B')
    tickers = [f'STOCK_{i:03d}' for i in range(n_stocks)]
    
    data = []
    for ticker in tickers:
        # Generate price series with trend and noise
        base_price = 50 + np.random.randn() * 20
        returns = np.random.randn(n_days) * 0.02 + 0.0002
        prices = base_price * np.exp(np.cumsum(returns))
        
        # Add volume with correlation to price movements
        volume = np.abs(np.random.randn(n_days) * 1e6 + 5e6)
        volume *= (1 + 0.3 * np.abs(returns) / np.std(returns))
        
        for i, date in enumerate(dates):
            # Introduce missing data (2% chance)
            if np.random.rand() > 0.02:
                data.append({
                    'date': date,
                    'ticker': ticker,
                    'close': prices[i],
                    'open': prices[i] * (1 + np.random.randn() * 0.005),
                    'high': prices[i] * (1 + abs(np.random.randn() * 0.01)),
                    'low': prices[i] * (1 - abs(np.random.randn() * 0.01)),
                    'volume': volume[i],
                })
    
    df = pd.DataFrame(data)
    
    # Introduce a few outliers (extreme prices)
    outlier_idx = np.random.choice(len(df), size=int(len(df) * 0.001), replace=False)
    df.loc[outlier_idx, 'close'] *= np.random.choice([0.1, 10], size=len(outlier_idx))
    
    return df.set_index(['date', 'ticker']).sort_index()

# Generate data
market_data = generate_sample_market_data(n_stocks=50, n_days=252)

display_dataframe_summary(market_data, "Raw Market Data")
print(f"\nDate range: {market_data.index.get_level_values(0).min()} to {market_data.index.get_level_values(0).max()}")
print(f"Unique tickers: {market_data.index.get_level_values(1).nunique()}")
print(f"Missing values: {market_data.isna().sum().sum()}")

## Step 3: Data Quality Checks

Before running factors, check for common data quality issues.

In [ ]:
def check_data_quality(df):
    """Run data quality checks on market data."""
    issues = {}
    
    # Check for missing values
    missing = df.isna().sum()
    if missing.sum() > 0:
        issues['missing_values'] = missing[missing > 0].to_dict()
    
    # Check for negative prices
    price_cols = ['close', 'open', 'high', 'low']
    for col in price_cols:
        if col in df.columns:
            neg_count = (df[col] <= 0).sum()
            if neg_count > 0:
                issues[f'negative_{col}'] = neg_count
    
    # Check for outliers (beyond 5 std devs from mean)
    for col in price_cols:
        if col in df.columns:
            z_scores = np.abs((df[col] - df[col].mean()) / df[col].std())
            outlier_count = (z_scores > 5).sum()
            if outlier_count > 0:
                issues[f'outliers_{col}'] = outlier_count
    
    # Check for high/low consistency
    if 'high' in df.columns and 'low' in df.columns and 'close' in df.columns:
        invalid = ((df['high'] < df['low']) | 
                   (df['close'] > df['high']) | 
                   (df['close'] < df['low'])).sum()
        if invalid > 0:
            issues['invalid_ohlc'] = invalid
    
    return issues

quality_issues = check_data_quality(market_data)

if quality_issues:
    display_warning(f"Found {len(quality_issues)} types of data quality issues")
    for issue, count in quality_issues.items():
        print(f"  {issue}: {count}")
else:
    display_success("No data quality issues found")

## Step 4: Clean and Preprocess Data

Apply cleaning transformations to handle the identified issues.

In [ ]:
def clean_market_data(df):
    """Clean market data by handling missing values and outliers."""
    df = df.copy()
    
    # Remove negative prices
    price_cols = ['close', 'open', 'high', 'low']
    for col in price_cols:
        if col in df.columns:
            df = df[df[col] > 0]
    
    # Cap outliers at 5 standard deviations
    for col in price_cols:
        if col in df.columns:
            mean = df[col].mean()
            std = df[col].std()
            lower = mean - 5 * std
            upper = mean + 5 * std
            df[col] = df[col].clip(lower, upper)
    
    # Forward fill missing values within each ticker
    df = df.groupby(level='ticker').ffill(limit=5)
    
    # Drop remaining missing values
    df = df.dropna()
    
    return df

with NotebookTimer("Data cleaning"):
    cleaned_data = clean_market_data(market_data)

print(f"\nOriginal rows: {len(market_data)}")
print(f"Cleaned rows:  {len(cleaned_data)}")
print(f"Dropped:       {len(market_data) - len(cleaned_data)} ({(len(market_data) - len(cleaned_data))/len(market_data)*100:.2f}%)")

# Verify cleaning
quality_after = check_data_quality(cleaned_data)
if not quality_after:
    display_success("All data quality issues resolved")
else:
    display_warning(f"Remaining issues: {quality_after}")

## Step 5: Universe Filtering

Filter stocks based on liquidity, market cap, or other criteria.

In [ ]:
def filter_universe(df, min_days=200, min_avg_volume=1e6):
    """Filter universe based on data availability and liquidity."""
    # Count trading days per ticker
    days_per_ticker = df.groupby(level='ticker').size()
    valid_tickers = days_per_ticker[days_per_ticker >= min_days].index
    
    df = df.loc[(slice(None), valid_tickers), :]
    
    # Filter by average volume
    if 'volume' in df.columns:
        avg_volume = df.groupby(level='ticker')['volume'].mean()
        liquid_tickers = avg_volume[avg_volume >= min_avg_volume].index
        df = df.loc[(slice(None), liquid_tickers), :]
    
    return df

filtered_data = filter_universe(cleaned_data, min_days=200, min_avg_volume=2e6)

print(f"Tickers before filtering: {cleaned_data.index.get_level_values(1).nunique()}")
print(f"Tickers after filtering:  {filtered_data.index.get_level_values(1).nunique()}")
print(f"\nFinal universe size: {len(filtered_data)} observations")

display_success("Universe filtering complete")

## Step 6: Create Custom DataSource

Wrap your cleaned data in a DataSource for use with the engine.

In [ ]:
class PandasDataSource(DataSource):
    """DataSource backed by a pandas DataFrame."""
    
    def __init__(self, df: pd.DataFrame):
        self.df = df
    
    def load_column(self, name: str) -> pd.Series:
        """Load a column as MultiIndex Series."""
        if name not in self.df.columns:
            raise KeyError(f"Column '{name}' not found. Available: {list(self.df.columns)}")
        return self.df[name]
    
    def get_dates(self):
        """Get unique dates in the dataset."""
        return sorted(self.df.index.get_level_values(0).unique())
    
    def get_tickers(self):
        """Get unique tickers in the dataset."""
        return sorted(self.df.index.get_level_values(1).unique())

# Create data source
data_source = PandasDataSource(filtered_data)

print(f"DataSource ready with {len(data_source.get_dates())} dates and {len(data_source.get_tickers())} tickers")
print(f"Available columns: {list(filtered_data.columns)}")

## Step 7: Run Factor on Cleaned Data

Now execute a factor using the preprocessed real data.

In [ ]:
# Create engine with real data
engine = FactorEngine(
    backend=PandasBackend(),
    data_source=data_source
)

# Define a momentum factor
momentum_expr = rank(ts_mean(col("close"), 20) - ts_mean(col("close"), 5))
momentum_factor = Factor("momentum_20_5", momentum_expr, "1d", "equities")

# Run the factor
with NotebookTimer("Factor calculation"):
    result = engine.run(momentum_factor)

factor_values = result["result"]

print(f"\nFactor output shape: {factor_values.shape}")
print(f"Non-null values: {factor_values.notna().sum()}")
print(f"Coverage: {factor_values.notna().sum() / len(factor_values) * 100:.1f}%")

# Show statistics
metrics = {
    'mean': factor_values.mean(),
    'std': factor_values.std(),
    'skew': factor_values.skew(),
    'kurtosis': factor_values.kurtosis(),
}

display_metrics(metrics, "Factor Distribution")

## Key Takeaways

1. **Data Quality**: Always check for missing values, outliers, and inconsistencies before calculation
2. **Preprocessing**: Clean systematically - handle negatives, cap outliers, forward fill cautiously
3. **Universe Filtering**: Filter by liquidity and data availability to ensure robust factors
4. **Custom DataSource**: Wrap your data in a DataSource interface for engine compatibility
5. **Coverage Monitoring**: Track how much of your universe has valid factor values

## Next Steps

- Tutorial 03: Factor optimization and parameter tuning
- Tutorial 04: Multi-factor selection strategies
- Examples: Real-world data integration patterns